# SlalomSkiing

[Codility task](https://app.codility.com/programmers/trainings/1/slalom_skiing/) - Exercise 1, 2015 Contest

Gate `K` is at distance `K + 1` from the start and `A[K]` from the right barrier. Skiing left passes gates with **increasing** `A`, skiing right passes gates with **decreasing** `A`. You start going left and may change direction at most twice. `solution(A)` returns the maximum number of gates you can pass.

| A | result |
|---|--------|
| `[15, 13, 5, 7, 4, 10, 12, 8, 2, 11, 6, 9, 3]` | 8 (`5 7 10 12`, then `8 2`, then `6 9`) |
| `[1, 5]` | 2 |

Assumptions: `N` in `[1..100,000]`, values in `[1..10^9]`, all distinct.

## Idea

The gates passed form a subsequence of `A` that goes **up, down, up** (each phase may contain a single gate). Three states per gate `i`, meaning "the best course ending at gate `i` in phase 1, 2 or 3":

- `f0[i] = 1 + max f0[j]` over `j < i` with `A[j] < A[i]`;
- `f1[i] = 1 + max(f0[j], f1[j])` over `j < i` with `A[j] > A[i]` (turn right after gate `j`);
- `f2[i] = 1 + max(f1[j], f2[j])` over `j < i` with `A[j] < A[i]` (turn left after gate `j`).

The answer is the largest `f` value. Evaluated directly this is O(N^2); with the values compressed to ranks, each "max over earlier gates with a smaller (or larger) value" is a **prefix maximum** query on a Fenwick tree, giving O(N log N). A value `0` means "state not reachable".

In [ ]:
class MaxFenwick:
    """Fenwick tree for prefix maxima over positions 1..n (values only grow)."""

    def __init__(self, n):
        self.n = n
        self.tree = [0] * (n + 1)

    def update(self, i, value):
        while i <= self.n:
            if self.tree[i] < value:
                self.tree[i] = value
            i += i & -i

    def query(self, i):
        best = 0
        while i > 0:
            if self.tree[i] > best:
                best = self.tree[i]
            i -= i & -i
        return best


def solution(A):
    n = len(A)
    rank = {value: r for r, value in enumerate(sorted(A), start=1)}
    smaller_f0 = MaxFenwick(n)          # indexed by rank: best f0 of earlier gates
    larger_g1 = MaxFenwick(n)           # indexed by n + 1 - rank: best max(f0, f1)
    smaller_g2 = MaxFenwick(n)          # indexed by rank: best max(f1, f2)

    best = 0
    for value in A:
        r = rank[value]
        f0 = smaller_f0.query(r - 1) + 1
        q1 = larger_g1.query(n - r)     # earlier gates with a larger value
        f1 = q1 + 1 if q1 else 0
        q2 = smaller_g2.query(r - 1)
        f2 = q2 + 1 if q2 else 0

        smaller_f0.update(r, f0)
        larger_g1.update(n + 1 - r, max(f0, f1))
        smaller_g2.update(r, max(f1, f2))
        best = max(best, f0, f1, f2)
    return best

## Official examples

In [ ]:
assert solution([15, 13, 5, 7, 4, 10, 12, 8, 2, 11, 6, 9, 3]) == 8
assert solution([1, 5]) == 2
print("examples OK")

## Edge cases

In [ ]:
assert solution([7]) == 1
assert solution([1, 2, 3, 4, 5]) == 5               # only the first direction
assert solution([5, 4, 3, 2, 1]) == 5               # one gate to the left (the first), then turn right: 1 change
assert solution([5, 1, 3]) == 3                     # right then left: 2 changes counting the initial left
assert solution([3, 1, 5, 2]) == 3                  # all four would need 3 changes
assert solution([1, 3, 2, 4]) == 4                  # up, down, up with single-gate middle phase
assert solution([2, 1, 4, 3]) == 3
print("edge cases OK")

## Brute force over all subsets

For small `N` every subset of gates is tried. The direction changes of a subset are counted as follows: the course *starts* going left, so a first step downwards (decreasing `A`) already costs one change; every later sign change of the steps costs another. A subset is feasible with at most two changes.

In [ ]:
import random


def brute(A):
    n = len(A)
    best = 0
    for mask in range(1, 1 << n):
        values = [A[i] for i in range(n) if mask >> i & 1]
        signs = [1 if b > a else -1 for a, b in zip(values, values[1:])]
        changes = (1 if signs and signs[0] < 0 else 0) + sum(1 for s, t in zip(signs, signs[1:]) if s != t)
        if changes <= 2:
            best = max(best, len(values))
    return best


assert brute([15, 13, 5, 7, 4, 10, 12, 8, 2, 11, 6, 9, 3]) == 8
zigzag = [10, 1, 9, 2, 8, 3, 7, 4, 6, 5]                  # alternating high / low values
assert solution(zigzag) == brute(zigzag)
print("zig-zag of 10 gates:", solution(zigzag))
rng = random.Random(4)
for _ in range(500):
    n = rng.randint(1, 11)
    A = rng.sample(range(1, 40), n)
    assert solution(A) == brute(A), A
print("matches the subset brute force on 500 random permutations (N up to 11)")

## Quadratic dynamic programming

The same three-state recurrence computed naively in O(N^2), compared on larger inputs than the brute force can handle.

In [ ]:
def quadratic(A):
    n = len(A)
    f0 = [1] * n
    f1 = [0] * n
    f2 = [0] * n
    for i in range(n):
        for j in range(i):
            if A[j] < A[i]:
                f0[i] = max(f0[i], f0[j] + 1)
                if f1[j] or f2[j]:
                    f2[i] = max(f2[i], max(f1[j], f2[j]) + 1)
            else:
                f1[i] = max(f1[i], max(f0[j], f1[j]) + 1)
    return max(max(f0), max(f1), max(f2))


for _ in range(300):
    n = rng.randint(1, 120)
    A = rng.sample(range(1, 10**9), n)
    assert solution(A) == quadratic(A), A
for _ in range(100):
    n = rng.randint(50, 200)
    A = rng.sample(range(1, 4 * n), n)            # dense values: many equal-rank interactions
    assert solution(A) == quadratic(A), A
print("matches the quadratic DP on 400 random inputs (N up to 200)")

## Performance

In [ ]:
import time

N = 100_000
cases = {
    "random": rng.sample(range(1, 10**9), N),
    "increasing": list(range(1, N + 1)),
    "decreasing": list(range(N, 0, -1)),
    "zig-zag": [i // 2 + 1 if i % 2 == 0 else N - i // 2 for i in range(N)],
    "three waves": list(range(1, N // 3 + 1)) + list(range(10**6, 10**6 - N // 3, -1)) + list(range(2 * 10**6, 2 * 10**6 + N - 2 * (N // 3))),
}
for name, A in cases.items():
    assert len(set(A)) == len(A)
    start = time.perf_counter()
    result = solution(A)
    print(f"{name:12s} -> {result:>7} in {time.perf_counter() - start:.2f}s")

## Notes

- Longest increasing/decreasing subsequence variants: when the phases are fixed (up, down, up) you keep **one DP state per phase**, and the transition between phases happens at the shared turning gate.
- The turning gate belongs to both phases, which is why `f1` reads `max(f0, f1)` of the previous gate.
- Fenwick trees answer "best value among earlier items with smaller key" in O(log N); the values only grow, so a max-Fenwick works.
- The initial direction is fixed (left), so a course that starts with a downward step already uses one change.